Cài PaddleOCR-VL

In [1]:
import os, shutil

VENV_DIR = "/kaggle/working/ppocrv6_env"

if os.path.exists(VENV_DIR):
    shutil.rmtree(VENV_DIR)

# Base Kaggle
!python -m pip install -U virtualenv

# Tạo environment sạch
!env -u PYTHONPATH -u PYTHONHOME -u PYTHONUSERBASE PYTHONNOUSERSITE=1 \
python -m virtualenv /kaggle/working/ppocrv6_env

VENV_PY = "/kaggle/working/ppocrv6_env/bin/python"

# Update build tools
!env -u PYTHONPATH -u PYTHONHOME -u PYTHONUSERBASE PYTHONNOUSERSITE=1 \
{VENV_PY} -m pip install -U pip setuptools wheel

# Paddle GPU CUDA 12.6
!env -u PYTHONPATH -u PYTHONHOME -u PYTHONUSERBASE PYTHONNOUSERSITE=1 \
{VENV_PY} -m pip install paddlepaddle-gpu==3.2.1 \
-i https://www.paddlepaddle.org.cn/packages/stable/cu126/

# PaddleOCR
!env -u PYTHONPATH -u PYTHONHOME -u PYTHONUSERBASE PYTHONNOUSERSITE=1 \
{VENV_PY} -m pip install -U "paddleocr>=3.7.0"

# Transformers backend + dependencies
!env -u PYTHONPATH -u PYTHONHOME -u PYTHONUSERBASE PYTHONNOUSERSITE=1 \
{VENV_PY} -m pip install -U \
transformers \
safetensors \
huggingface_hub \
accelerate

# Basic image libs
!env -u PYTHONPATH -u PYTHONHOME -u PYTHONUSERBASE PYTHONNOUSERSITE=1 \
{VENV_PY} -m pip install -U \
pillow \
opencv-python \
numpy

ERROR: Could not find a version that satisfies the requirement virtualenv (from versions: none)
ERROR: No matching distribution found for virtualenv
/usr/bin/python3: No module named virtualenv
env: ‘/kaggle/working/ppocrv6_env/bin/python’: No such file or directory
env: ‘/kaggle/working/ppocrv6_env/bin/python’: No such file or directory
env: ‘/kaggle/working/ppocrv6_env/bin/python’: No such file or directory
env: ‘/kaggle/working/ppocrv6_env/bin/python’: No such file or directory
env: ‘/kaggle/working/ppocrv6_env/bin/python’: No such file or directory


In [2]:
from pathlib import Path

INPUT_ROOT = Path(
    "/kaggle/input/datasets/diamondzone/aic-batch1/L21a/kaggle/working/data/keyframe"
)

# Output lưu trên Google Drive
OUTPUT_ROOT = Path(
    "/kaggle/working/test"
)

# File ZIP kết quả cuối cùng
OUTPUT_ZIP = Path(
    "/kaggle/working/test.zip"
)
# ============================================================

assert INPUT_ROOT.exists(), f"Không tìm thấy INPUT_ROOT: {INPUT_ROOT}"

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print("Input root :", INPUT_ROOT)
print("Output root:", OUTPUT_ROOT)
print("Output ZIP :", OUTPUT_ZIP)

Input root : /kaggle/input/datasets/diamondzone/aic-batch1/L21a/kaggle/working/data/keyframe
Output root: /kaggle/working/test
Output ZIP : /kaggle/working/test.zip


In [3]:
!env \
-u PYTHONPATH \
-u PYTHONHOME \
-u PYTHONUSERBASE \
-u MPLBACKEND \
PYTHONNOUSERSITE=1 \
"$VENV_PY" \
/kaggle/working/run_ppocrv6.py \
--input_root "{INPUT_ROOT}" \
--output_root "{OUTPUT_ROOT}" \
--zip_path "{OUTPUT_ZIP}"

env: ‘/kaggle/working/ppocrv6_env/bin/python’: No such file or directory


In [4]:
%%writefile /kaggle/working/run_ppocrv6.py

import os
import re
import time
import shutil
import argparse
import traceback
import multiprocessing as mp
from pathlib import Path


# ============================================================
# ARGUMENTS
# ============================================================

parser = argparse.ArgumentParser()
parser.add_argument("--input_root", required=True)
parser.add_argument("--output_root", required=True)
parser.add_argument("--zip_path", required=True)
parser.add_argument("--overwrite", action="store_true")
parser.add_argument("--max_images", type=int, default=None)
args = parser.parse_args()

INPUT_ROOT = Path(args.input_root)
OUTPUT_ROOT = Path(args.output_root)
ZIP_PATH = Path(args.zip_path)

IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"}


# ============================================================
# HELPERS
# ============================================================

def ensure_dir(path):
    path.mkdir(parents=True, exist_ok=True)
    return path


def natural_sort_key(path):
    parts = re.split(r"(\d+)", path.name.lower())
    return [int(part) if part.isdigit() else part for part in parts]


def make_unique_image_folder_name(image_path, source_folder):
    relative_stem = image_path.relative_to(source_folder).with_suffix("")
    return "__".join(relative_stem.parts)


def output_is_complete(image_output_dir, base_name):
    if not image_output_dir.exists():
        return False

    json_path = image_output_dir / f"{base_name}.json"
    img_path = image_output_dir / f"{base_name}.png"

    return json_path.exists() and img_path.exists()


# ============================================================
# WORKER
# ============================================================

def worker(physical_gpu_id, jobs, output_root, overwrite, result_queue, progress_counter):

    # Mỗi process chỉ nhìn thấy 1 GPU vật lý
    os.environ["CUDA_VISIBLE_DEVICES"] = str(physical_gpu_id)
    os.environ.pop("MPLBACKEND", None)
    os.environ["MPLBACKEND"] = "Agg"
    os.environ.setdefault("FLAGS_use_cuda_managed_memory", "true")

    # Import sau khi set CUDA_VISIBLE_DEVICES
    from paddleocr import PaddleOCR

    # PP-OCRv6 Medium
    ocr = PaddleOCR(
        text_detection_model_name="PP-OCRv6_medium_det",
        text_recognition_model_name="PP-OCRv6_medium_rec",
        use_doc_orientation_classify=False,
        use_doc_unwarping=False,
        use_textline_orientation=False,
        text_recognition_batch_size=16,
        device="gpu:0",
        enable_mkldnn=False,
    )

    success_count = 0
    error_count = 0
    skipped_count = 0

    error_log_path = Path(output_root).parent / f"ppocrv6_errors_gpu{physical_gpu_id}.txt"

    if error_log_path.exists():
        error_log_path.unlink()

    for source_folder_str, image_path_str in jobs:
        source_folder = Path(source_folder_str)
        image_path = Path(image_path_str)

        # Output:
        # L27_V001/
        #   keyframe_14220/
        #       keyframe_14220.json
        #       keyframe_14220.png
        folder_output = ensure_dir(Path(output_root) / source_folder.name)
        image_folder_name = make_unique_image_folder_name(image_path, source_folder)
        image_output_dir = ensure_dir(folder_output / image_folder_name)

        json_path = image_output_dir / f"{image_path.stem}.json"
        img_path = image_output_dir / f"{image_path.stem}.png"

        # Skip nếu đã có đủ JSON + ảnh
        if output_is_complete(image_output_dir, image_path.stem) and not overwrite:
            skipped_count += 1
            with progress_counter.get_lock():
                progress_counter.value += 1
            continue

        # Xóa output cũ nếu overwrite
        if overwrite:
            if json_path.exists():
                json_path.unlink()
            if img_path.exists():
                img_path.unlink()

        try:
            results = ocr.predict(str(image_path))
            result_found = False

            for res in results:
                result_found = True
                res.save_to_json(save_path=str(json_path))
                res.save_to_img(save_path=str(img_path))

            if not result_found:
                raise RuntimeError("PaddleOCR returned no result.")

            if not json_path.exists() or not img_path.exists():
                raise RuntimeError("Output JSON or image was not created.")

            success_count += 1

        except Exception as error:
            error_count += 1

            with open(error_log_path, "a", encoding="utf-8") as f:
                f.write("=" * 80 + "\n")
                f.write(f"GPU: {physical_gpu_id}\n")
                f.write(f"Image: {image_path}\n")
                f.write(f"Error: {error}\n")
                f.write(traceback.format_exc())
                f.write("\n\n")

        finally:
            with progress_counter.get_lock():
                progress_counter.value += 1

    result_queue.put({
        "gpu": physical_gpu_id,
        "success": success_count,
        "errors": error_count,
        "skipped": skipped_count,
    })


# ============================================================
# MAIN
# ============================================================

def main():
    from tqdm import tqdm

    if not INPUT_ROOT.exists():
        raise FileNotFoundError(f"Input root does not exist: {INPUT_ROOT}")

    ensure_dir(OUTPUT_ROOT)
    ensure_dir(ZIP_PATH.parent)

    # Tìm các folder video: L27_V001, L27_V002, ...
    source_folders = sorted(
        [p for p in INPUT_ROOT.iterdir() if p.is_dir()],
        key=natural_sort_key
    )

    if not source_folders:
        raise RuntimeError(f"No child folders found inside: {INPUT_ROOT}")

    # Collect toàn bộ ảnh
    jobs = []

    for source_folder in source_folders:
        image_files = sorted(
            [
                p for p in source_folder.rglob("*")
                if p.is_file() and p.suffix.lower() in IMAGE_EXTS
            ],
            key=natural_sort_key
        )

        for image_path in image_files:
            jobs.append((str(source_folder), str(image_path)))

    if args.max_images is not None:
        jobs = jobs[:args.max_images]

    total_jobs = len(jobs)

    if total_jobs == 0:
        raise RuntimeError("No images found.")

    print(f"Images: {total_jobs}")
    print(f"Output: {OUTPUT_ROOT}")

    # Chia xen kẽ cho 2 GPU
    gpu0_jobs = jobs[0::2]
    gpu1_jobs = jobs[1::2]

    ctx = mp.get_context("spawn")
    result_queue = ctx.Queue()
    progress_counter = ctx.Value("i", 0)

    p0 = ctx.Process(
        target=worker,
        args=(0, gpu0_jobs, str(OUTPUT_ROOT), args.overwrite, result_queue, progress_counter)
    )

    p1 = ctx.Process(
        target=worker,
        args=(1, gpu1_jobs, str(OUTPUT_ROOT), args.overwrite, result_queue, progress_counter)
    )

    p0.start()
    p1.start()

    # Một progress bar chung cho cả 2 GPU
    with tqdm(
        total=total_jobs,
        desc="PP-OCRv6",
        unit="image",
        dynamic_ncols=True,
        leave=True,
        mininterval=1.0,
        smoothing=0.1,
    ) as bar:

        last_value = 0

        while p0.is_alive() or p1.is_alive():
            current_value = progress_counter.value

            if current_value > last_value:
                bar.update(current_value - last_value)
                last_value = current_value

            time.sleep(0.5)

        current_value = progress_counter.value

        if current_value > last_value:
            bar.update(current_value - last_value)

    p0.join()
    p1.join()

    if p0.exitcode != 0:
        raise RuntimeError(f"GPU0 worker failed: exitcode={p0.exitcode}")

    if p1.exitcode != 0:
        raise RuntimeError(f"GPU1 worker failed: exitcode={p1.exitcode}")

    results = [result_queue.get(), result_queue.get()]

    total_success = sum(r["success"] for r in results)
    total_errors = sum(r["errors"] for r in results)
    total_skipped = sum(r["skipped"] for r in results)

    # ZIP output
    if ZIP_PATH.exists():
        ZIP_PATH.unlink()

    shutil.make_archive(
        base_name=str(ZIP_PATH.with_suffix("")),
        format="zip",
        root_dir=str(OUTPUT_ROOT.parent),
        base_dir=OUTPUT_ROOT.name,
    )

    print(
        f"DONE | Images={total_jobs} | Success={total_success} | "
        f"Skipped={total_skipped} | Errors={total_errors} | ZIP={ZIP_PATH}"
    )


if __name__ == "__main__":
    main()

Writing /kaggle/working/run_ppocrv6.py


In [5]:
from IPython.display import FileLink

FileLink("/kaggle/working/test.zip")

/kaggle/working/test.zip